From Cross-Sections to Panels
=============================

**Author:** Ethan Ligon



Issues and opportunities that arise from working with longitudinal panels of household data.



## Reading



### Reading



Deaton is in your `reading/` folder on the hub, or [PDF](https://documents.worldbank.org/curated/en/203811547671768139/pdf/133790-PUB.pdf).

-   Deaton, ch. 2 §2.7 and ch. 6
-   Mundlak (1961), "Empirical production function free of management bias" *J. of Farm Economics* 43(1):44-56.
-   Imbens & Wooldridge (2009), "Linear Panel Data Models," cemmap lectures
    3–4 ([slides I](https://cemmap.ac.uk/wp-content/legacy/resources/imbens_wooldridge/slides_3.pdf),
    [slides II](https://cemmap.ac.uk/wp-content/legacy/resources/imbens_wooldridge/slides_4.pdf),
    [notes](https://cemmap.ac.uk/wp-content/legacy/resources/imbens_wooldridge/lecture_34.pdf)).
    Notes sections 1, 3, and 6: fixed effects and first differences,
    exogeneity, and pseudo-panels.



### Reading: cohorts and inference



-   Deaton (1985), "Panel data from time series of cross-sections"
-   Deaton & Paxson (1994), "Intertemporal Choice and Inequality,"
    *J. of Political Economy* 102(3):437–467
    ([paper](https://www.princeton.edu/~deaton/downloads/Intertemporal_Choice_and_Inequality.pdf)). Section II: following birth cohorts and their consumption dispersion.
-   Abadie, Athey, Imbens & Wooldridge (2023), "When should you adjust
    standard errors for clustering?" *QJE* 138:1–35



## Longitudinal Panels



### Introduction to Panels



### Panels in `lsms_library`



The library's `panel_ids` crosswalks connect wave-specific identifiers.
The returned index level `i` uses those links; matching raw household
numbers ourselves can join unrelated households.



#### Preface



In [1]:
# Show tracebacks without the library's internal frames: the line that
# failed, and why.  Change Plain to Verbose if you ever want the rest.
%xmode Plain

# The library audits its own corpus on first read and reports what it finds
# --- implausible quantities, NaN index keys, a column that is wholly null in
# one wave --- at multi-paragraph length.  Those reports are a work queue for
# whoever maintains the data, not something the room can act on, and they bury
# the output they are attached to.  Silenced here by their own
# "Set LSMS_..._STRICT=1" signature, which is precise: every other warning,
# pandas deprecations included, still shows.  Delete these two lines to read
# them.
import warnings
warnings.filterwarnings("ignore",
                        message=r"(?s).*Set LSMS_[A-Z_]+=1 to make this fatal")

# The first call that builds a table may print "DVC unavailable ...
# falling back to manual aggregation".  That is about how the library
# fetches its raw files, not about your data; the numbers are the same,
# and it does not recur once the table is built.

import matplotlib as mpl
mpl.rcParams.update({'figure.dpi': 110, 'font.size': 11,
                     'lines.linewidth': 1.8})

import lsms_library as ll
import numpy as np
import pandas as pd
from IPython.display import display

#### Which surveys declare a crosswalk



A survey's `data_scheme` says what the library knows how to build for it.
Declaring `panel_ids` is the survey saying that its waves can be linked at
the household, and that someone has written down how.

Find those surveys first. This reads the catalogue only: no household
records are loaded, so it is quick and it tells us where a panel is even
possible.



In [1]:
panel_countries = {}
for name in ll.countries():
    country = ll.Country(name)
    if 'panel_ids' in country.data_scheme:
        panel_countries[name] = country
print(", ".join(panel_countries))

The following snapshot uses extracts checked on 17 September 2026.
*Waves* counts the survey's available `t` labels, and the last two
columns give the smallest and largest *whole-wave* household counts in
`sample()`. They are not the size of a balanced panel.


| Survey|Waves|Minimum households|Maximum households|
|---|---|---|---|
| Burkina_Faso|3|3,227|10,411|
| Ethiopia|5|3,969|6,770|
| EthiopiaRHS|8|338|1,480|
| GhanaLSS|7|3,147|16,772|
| Malawi|5|4,000|14,955|
| Mali|4|3,804|8,390|
| Niger|4|3,558|6,622|
| Nigeria|10|4,696|5,263|
| Senegal|2|7,120|7,156|
| Tanzania|6|1,184|5,010|
| Uganda|8|2,716|3,305|

Nigeria's ten labels are planting/harvest visits in five survey waves.
Only GhanaLSS's first two waves are linked. Ethiopia and Niger contain
separate panel cohorts; Tanzania has extended and refreshment panels.
A country's full list of waves is not one continuously followed sample.

Recompute the table from your installed library. We count distinct
households, not rows of an expenditure or person-level table.



In [1]:
inventory_rows = []
for name, country in panel_countries.items():
    sample_counts = (country.sample().reset_index()
                    .groupby('t')['i'].nunique())
    inventory_rows.append({
        'Survey': name,
        'Waves listed': len(country.waves),
        'Waves in sample': len(sample_counts),
        'Minimum households': sample_counts.min(),
        'Maximum households': sample_counts.max(),
    })
panel_inventory = pd.DataFrame(inventory_rows).set_index('Survey')
display(panel_inventory)

The catalogue also contains these documented panel components without a
declared `panel_ids` crosswalk. They require further linkage work before
the same exercise is justified; the absence of a library crosswalk does
not mean the survey was a repeated cross-section.


| Survey|Panel component|
|---|---|
| Albania|2002–04; three waves|
| CotedIvoire|1985–88; four rounds, rotating two-year panels|
| GhanaSPS|2009-10, 2013-14, 2017-18; three waves|
| Nepal|1995-96, 2003-04, 2010-11; panel subsamples|
| Peru|Partial links 1985–90 and 1991–94; coverage varies|
| Serbia and Montenegro|2002–03; two waves|
| Tajikistan|2007–09; two waves|



### Why two sets of effects



### Two-way Fixed Effects Regressions



### TWFE generalizes a double difference



### The within transformation



### You have already used one with different slopes



### Where the causal content comes from



### What makes the within comparison valid?



### What fixed effects can remove



### Attrition



### Attrition and consistency



### A genuine panel: Uganda



### Observed Attrition in the Uganda Panel



One call puts the whole panel in front of you. The diagonal counts the
households in that wave; a cell above it counts the IDs the two waves
share. Sharing is pairwise — it does not require presence in every
wave in between.



In [1]:
ll.Country('Uganda').panel_attrition()

Read the first row, then the diagonal. Of the 3,123 households
interviewed in 2005–06, 1,291 are still there in 2019–20: fewer than
half, over fourteen years. Yet the diagonal never falls below 2,714.
The survey is the same size throughout because it is being refreshed,
and from 2013–14 the refreshment is large enough to see in the first
row, which drops from 2,368 to 1,568 at that wave.

So a wave's sample size tells you nothing about how many households you
can follow, and "the Uganda panel" is not one sample followed for
fourteen years. Take the sub-matrix you can actually use before you
count on any of it.

The count is of exact IDs. With `split_households_new_sample=False` the
library also counts later descendant households; a parent can have
several, so that count divided by the initial sample size is not a
retention probability.

`Country.panel_attrition()` reads the household roster. To ask the same
question of the table you will actually use — expenditures, say —
pass that dataframe to `ll.tools.panel_attrition` instead.



### Removing Fixed Effects: a balanced panel



### Removing Fixed Effects: an unbalanced panel



### A GLSS panel



### Examine the GLSS Panel



The same call, for Ghana.



In [1]:
ghana = ll.Country('GhanaLSS')
ghana.panel_attrition()

One pair of waves shares households and no other pair shares any: 712
IDs appear in both 1987–88 and 1988–89, and every other off-diagonal
cell is zero. Those zeros are not attrition. They are the absence of a
crosswalk — `PANELC.DAT` was published for those two rounds and for no
others — and a survey that cannot be linked looks exactly like a
survey nobody stayed in.

712 is also fewer than the 714 distinct GLSS2 households in the raw
person crosswalk. Descendants carry their own IDs and are not the same
unit as their parent; do not collapse them to force a larger panel.

Describe them. Reshaping to one row per household with a column per
wave, then dropping the rows with a gap, *is* the balancing operation:
what survives has both waves by construction.



In [1]:
(np.exp(ghana.household_characteristics(waves=['1987-88', '1988-89'])['log HSize'])
 .droplevel('v').unstack('t').dropna().describe())

All 712 survive the `dropna`, so every linked household has a roster in
both waves, and mean size goes from 5.33 to 5.50. That describes the
households we can follow, unweighted. It is not an estimate of how
Ghanaian households changed: the ones we can follow are the half of
GLSS1's workloads that were kept on, chosen by design and not by who
agreed to answer again.

Two waves and 712 households is what the rest of this session has to
work with, and the application below will ask for rather more than a
household ID.



## Application: Farm Management in Ghana



### Management and measured productivity



### Cobb–Douglas: technology and management



### From output to a linear revenue equation



### What the Ghana tables observe



### Construct the household-year observations



The five Ghana farm tables ship in `lsms_library` 0.14.0, and cover
1987-88 and 1988-89 only; an earlier release cannot run this section.

The construction follows the Mundlak how-to in your `reading/` folder
([HOWTO\_MUNDLAK.pdf](https://hhsurveys.ligonresearch.org/hub/user-redirect/files/reading/HOWTO_MUNDLAK.pdf)), which carries the diagnostics behind the choices
made here and the ones it decided against.

Each table has more detail than our household regression needs. Aggregate
by the named `t` and `i` levels, retaining a wholly missing sum as missing.
This also retains records whose cluster identifier is missing.



In [1]:
import warnings
warnings.filterwarnings("ignore",
                        message=r"(?s).*Set LSMS_[A-Z_]+=1 to make this fatal")
import numpy as np
import pandas as pd
import datamat as dm
import lsms_library as ll
from IPython.display import display

farm_country = ll.Country('GhanaLSS')
farm_waves = ['1987-88', '1988-89']
farm_features = ['crop_production', 'labor', 'plot_labor', 'plot_inputs']
assert all(name in farm_country.data_scheme for name in farm_features), (
    "The Ghana farm tables need lsms_library 0.14.0 or later")
print("Library source:", ll.__file__)

def farm_sum(series, name):
    return series.groupby(level=['t', 'i']).sum(min_count=1).rename(name)

farm_crop = farm_country.crop_production(waves=farm_waves)
farm_jobs = farm_country.labor(waves=farm_waves)
farm_work = farm_country.plot_labor(waves=farm_waves)
farm_inputs = farm_country.plot_inputs(waves=farm_waves)
farm_sample = farm_country.sample(waves=farm_waves)

First construct the dependent variable. Adding the four dispositions
produces a partial value, in nominal pre-2007 cedis. It mixes sale receipts
with replacement valuations and omits food consumed by the household.
The default API does not deflate these amounts. A common multiplicative
price change can enter a wave effect; household-specific prices cannot.

A partially observed sum remains partial. The coverage table counts
positive-area crop rows with no observed disposition value at all.



In [1]:
farm_value_columns = ['Value_sold', 'Value_seed', 'Value_given', 'Value_lost']
farm_crop_value = farm_crop[farm_value_columns].sum(axis=1, min_count=1)
farm_output = farm_sum(farm_crop_value, 'partial_value')
farm_area = farm_sum(farm_crop['Area_ha'], 'crop_area_ha')
positive_area = farm_crop['Area_ha'].gt(0)
coverage = pd.DataFrame({
    'positive_area_rows': positive_area,
    'no_value_rows': positive_area & farm_crop_value.isna(),
}).groupby('t').sum()
coverage['no_value_percent'] = (100 * coverage['no_value_rows']
                              / coverage['positive_area_rows'])
display(coverage.round(1))

### Put the three labor measures in common units



Family labor is annual weeks times usual hours per week. The industry
codes 100–199 are the HOWTO's working agriculture classification, not a
verified crop-only classification. They can include forestry and fishing.
The library combines annual information from the reference-week job
modules with supplementary annual jobs; repeated-job pointer rows carry
no additional hours and mustn't create extra labor.

For hired labor, divide expenditure by the median *day* wage of paid
agricultural workers in that wave. For family labor, divide hours by the
pooled median reported hours per day. Display the observations behind
both conversions. These are approximations to comparable labor services,
not measurements of identical workers or tasks.



In [1]:
farm_agri = farm_jobs['Industry'].between(100, 199).fillna(False).astype(bool)
farm_paid = (farm_agri & farm_jobs['OwnFarmOrBusiness'].eq(False).fillna(False).astype(bool)
             & farm_jobs['Earnings'].gt(0).fillna(False).astype(bool)
             & farm_jobs['EarningsUnit'].eq('Day').fillna(False).astype(bool))
farm_wage_summary = (farm_jobs.loc[farm_paid, 'Earnings'].groupby('t')
                     .agg(['count', 'median']))
farm_day_wage = farm_wage_summary['median']
farm_hours = farm_jobs.loc[farm_agri, 'HoursPerDay'].dropna()
farm_hours_per_day = farm_hours.median()
assert farm_day_wage.reindex(farm_waves).gt(0).all()
assert farm_hours_per_day > 0
print("Hours/day observations:", len(farm_hours),
      "; pooled median:", farm_hours_per_day)
display(farm_wage_summary)

farm_annual_hours = farm_jobs['WeeksPerYear'] * farm_jobs['HoursPerWeek']
farm_own = farm_agri & farm_jobs['OwnFarmOrBusiness'].eq(True).fillna(False).astype(bool)
farm_family_days = (farm_sum(farm_annual_hours.where(farm_own), 'family_days')
                    / farm_hours_per_day)
farm_source = farm_work.index.get_level_values('source')
farm_hired_cost = farm_sum(
    farm_work['Cost'].where(farm_source == 'hired'), 'hired_cost')
farm_exchange = farm_sum(
    farm_work['PersonDays'].where(farm_source == 'exchange'), 'exchange_days')
farm_data = pd.concat([farm_output, farm_area, farm_family_days,
                       farm_hired_cost, farm_exchange], axis=1)
farm_data['hired_days'] = (farm_data['hired_cost']
    / farm_data.index.get_level_values('t').map(farm_day_wage))

The HOWTO sums available family, hired, and exchange days, treating absent
components as zero only when at least one component is observed. The
library drops recorded zero hired/exchange cells, but absence alone can
also reflect missing information. We retain the component-availability
counts so this assumption is visible. It can understate labor where an
unobserved component was positive.

The country notes identify three marketing costs: transport,
containers, and storage. Keep them separate from purchased production
inputs. Neither input cost nor hired cost needs to be
positive for our main land-and-total-labor specification.



In [1]:
farm_parts = ['family_days', 'hired_days', 'exchange_days']
farm_data['labor_components_observed'] = farm_data[farm_parts].notna().sum(axis=1)
farm_data['labor_days'] = farm_data[farm_parts].sum(axis=1, min_count=1)
display(pd.crosstab(farm_data.index.get_level_values('t'),
                    farm_data['labor_components_observed']))
farm_marketing = farm_inputs.index.get_level_values('input').isin(
    ['Transport', 'Containers', 'Storage'])
farm_data['production_input_cost'] = farm_sum(
    farm_inputs['Cost'].where(~farm_marketing), 'production_input_cost')
farm_data['marketing_cost'] = farm_sum(
    farm_inputs['Cost'].where(farm_marketing), 'marketing_cost')
display(farm_data.groupby('t')[farm_parts + ['labor_days']].median().round(1))

### Define the estimation sample



Start from linked sample IDs, preserving the library's split-household
convention. The 712 linked households are where this begins, not where
it ends: what follows needs farms, and needs them measured twice.

Show overlap in the actual tables, then require finite, positive partial
value, crop area, and total labor in *both* waves. A missing disposition
isn't evidence of zero production. Taking logs and balancing select a
sample; neither operation repairs selection bias.



In [1]:
farm_overlap, farm_links = ll.tools.panel_attrition(
    farm_sample, farm_waves, return_ids=True,
    split_households_new_sample=True)
farm_linked_ids = farm_links[tuple(farm_waves)]
for name, table in [('sample', farm_sample), ('crops', farm_crop),
                    ('annual labor', farm_jobs), ('hired/exchange', farm_work)]:
    print(name)
    display(ll.tools.panel_attrition(table, farm_waves).astype('Int64'))

farm_columns = ['partial_value', 'crop_area_ha', 'labor_days']
farm_usable = farm_data[farm_columns].replace([np.inf, -np.inf], np.nan).dropna()
farm_usable = farm_usable[farm_usable.gt(0).all(axis=1)]
print("Usable household-years before linkage/balancing:", len(farm_usable))
farm_candidates = farm_usable[
    farm_usable.index.get_level_values('i').isin(farm_linked_ids)]
farm_nwaves = farm_candidates.groupby('i').size()
farm_ids = farm_nwaves.index[farm_nwaves.eq(2)]
farm_panel = farm_candidates[
    farm_candidates.index.get_level_values('i').isin(farm_ids)].sort_index()
assert farm_panel.index.is_unique
assert len(farm_panel) == 2 * len(farm_ids) and len(farm_ids) > 0
farm_log = np.log(farm_panel.astype(float))
print("Linked sample households:", len(farm_linked_ids))
print("Complete farm households:", len(farm_ids), "; panel rows:", len(farm_panel))
print("Lost from linked sample:", len(farm_linked_ids) - len(farm_ids))
display(farm_panel.groupby('t').agg(['count', 'mean', 'median']).round(2))
farm_input_spend = farm_data.loc[farm_panel.index, 'production_input_cost']
display(pd.DataFrame({'missing': farm_input_spend.isna(),
                      'zero': farm_input_spend.eq(0),
                      'positive': farm_input_spend.gt(0)}).groupby('t').sum())
print("Farms with positive production-input spend in both waves:",
      int(farm_input_spend.gt(0).fillna(False).groupby('i').all().sum()))

This additional positivity restriction would select a different farm
sample. Its count needn't match the HOWTO's purchased-input example,
which sums marketing and production costs together.



### Diagnose land before estimating



The land regressor sums harvested area across crops. Multiple cropping
and intercropping can make this different from physical land operated.
Changes in crop coverage or reporting can create changes in this sum.

Count positive-area crop records and compare crop sets across the two
waves. The diagnostic below reports the root mean squared within-farm
log-area deviation on the full panel and on farms with the same crop
set. It uses the same definition as the HOWTO's within standard deviation.
A stable crop set doesn't establish accurate area, and restricting to
it selects farms partly on a production choice.



In [1]:
farm_grown = farm_crop.loc[farm_crop['Area_ha'].gt(0)].reset_index()
farm_crop_sets = farm_grown.groupby(['t', 'i'])['j'].agg(frozenset)
farm_crop_counts = farm_crop_sets.map(len)
display(farm_crop_counts.groupby('t').describe().round(2))
farm_sets_wide = farm_crop_sets.unstack('t').reindex(farm_ids)
farm_same_crops = farm_sets_wide[farm_waves[0]].eq(farm_sets_wide[farm_waves[1]])
farm_stable_ids = farm_same_crops.index[farm_same_crops]
farm_land_diagnostics = []
for name, ids in [('all panel farms', farm_ids),
                  ('same crop set', farm_stable_ids)]:
    log_area = farm_log.loc[farm_log.index.get_level_values('i').isin(ids),
                           'crop_area_ha']
    within_area = log_area - log_area.groupby('i').transform('mean')
    farm_land_diagnostics.append({
        'sample': name, 'farms': len(ids),
        'within_log_area_rms': np.sqrt(within_area.pow(2).mean())})
display(pd.DataFrame(farm_land_diagnostics).set_index('sample').round(3))
farm_area_change = (farm_log['crop_area_ha'].unstack('t')[farm_waves[1]]
                    - farm_log['crop_area_ha'].unstack('t')[farm_waves[0]])
display(farm_area_change.describe(percentiles=[.1, .5, .9]).round(3))

At the checked feature revision, the within log-area RMS is 1.117 for
all 506 farms and 1.073 for the 21 farms with unchanged crop sets.
The median log-area change is 1.924. This diagnostic doesn't resolve
the area problem: holding the reported crop set fixed leaves large
changes, and does so on a very small, selected subsample.



### Cross-sectional estimation



### Estimate with DataMat



`DataMat.lstsq` solves least squares. It doesn't choose an intercept,
align two unrelated samples, or supply clustered uncertainty for us.
We provide the same ordered rows of outcome and regressors and check
rank before fitting.

For the later farm-mean test, add a farm-cluster sandwich covariance.
Sum the regressor-times-residual scores within each farm, then form
$$
\widehat V=\frac{G}{G-1}\frac{n-1}{n-k}
(X'X)^{-1}\left(\sum_i s_i s_i'\right)(X'X)^{-1}.
$$

Here $G$ counts farms, $n$ rows, and $k$ fitted coefficients.
This CR1 covariance permits dependence between a farm's two
observations. It assumes independent farms; dependence across farms in
a sampling cluster would require clustering at that level instead.

The estimator is not about farms, so we give it a neutral name and use it
again on the Uganda panel at the end of the session, where the cluster is
the household. `coef_table` prints a fit the way a regression table reads.



In [1]:
def clustered_ols(X, y):
    terms = X.columns
    X, y = dm.DataMat(X).astype(float), dm.DataVec(y).astype(float)
    assert X.index.equals(y.index)
    assert np.isfinite(X).all().all() and np.isfinite(y).all()
    n, k = X.shape
    assert X.rank() == k and n > k      # DataMat.rank is the matrix rank
    b = X.lstsq(y)                      # a DataVec, labelled by regressor
    scores = dm.DataMat(X.mul(y.resid(X), axis=0).groupby(level='i').sum())
    G = len(scores)
    assert G > 1
    # .dot(), not @: with a single regressor `@' squeezes the 1x1 product
    # to a vector and the labels go with it.
    bread = X.T.dot(X).inv()
    V = bread.dot(scores.T.dot(scores)).dot(bread)
    V = V * (G / (G - 1)) * ((n - 1) / (n - k))
    # DataMat labels a product with a one-level MultiIndex.  Hand back the
    # plain regressor names, so V.loc[term, term] is a number and not a
    # 1x1 table.  (Assigning .index instead leaves DataMat's own label
    # state behind, and dg() then disagrees with the index you can see.)
    return (pd.Series(np.asarray(b).reshape(-1), index=terms),
            pd.DataFrame(np.asarray(V), index=terms, columns=terms))

def coef_table(b, V):
    """Coefficients, cluster-robust standard errors, and t statistics."""
    se = pd.Series(np.sqrt(np.diag(V)), index=b.index)
    return pd.DataFrame({'coef': b, 'std err': se, 't': b / se}).round(3)

farm_xcols = ['crop_area_ha', 'labor_days']
farm_cross = {}
for wave in farm_waves:
    wave_data = farm_log.xs(wave, level='t').sort_index()
    X = wave_data[farm_xcols].assign(const=1.0)
    farm_cross[wave] = clustered_ols(X, wave_data['partial_value'])
farm_pooled_X = farm_log[farm_xcols].assign(
    second_wave=(farm_log.index.get_level_values('t') == farm_waves[1]).astype(float),
    const=1.0)
farm_pooled = clustered_ols(farm_pooled_X, farm_log['partial_value'])

### Two waves: remove persistent omitted factors



### Estimate changes and compare



With two complete waves, first differences with an intercept give the
TWFE slopes. Household demeaning alone leaves the time effect behind.
Subtract by household labels, checking that both waves contain the same
households in the same order.



In [1]:
farm_first = farm_log.xs(farm_waves[0], level='t').sort_index()
farm_second = farm_log.xs(farm_waves[1], level='t').sort_index()
assert farm_first.index.equals(farm_second.index)
farm_difference = farm_second - farm_first
farm_fd_X = farm_difference[farm_xcols].assign(const=1.0)
farm_fd = clustered_ols(farm_fd_X, farm_difference['partial_value'])
farm_models = {**farm_cross, 'pooled + wave': farm_pooled, 'first differences': farm_fd}
farm_comparison = pd.DataFrame({
    name: pd.Series({term: b[term] for term in farm_xcols})
    for name, (b, V) in farm_models.items()})
farm_se = pd.DataFrame({
    name: pd.Series({term: np.sqrt(V.loc[term, term]) for term in farm_xcols})
    for name, (b, V) in farm_models.items()})
print("Slopes; same", len(farm_ids), "farms in every model")
display(farm_comparison.round(3))
print("Farm-cluster CR1 standard errors")
display(farm_se.round(3))

A difference between the columns isn't an estimate of management ability.
It also reflects the different variation used, measurement error, and
possible departures from the common-slope model. Restricting to a common
sample removes one source of difference, not all of them.

Repeat the differenced fit for the same-crop-set subsample. Report its
size and compare coefficients; don't choose the sample because its
coefficient looks more plausible.



In [1]:
farm_stable = farm_difference.loc[farm_stable_ids].sort_index()
farm_stable_X = farm_stable[farm_xcols].assign(const=1.0)
if len(farm_stable) > len(farm_stable_X.columns) and (
        dm.DataMat(farm_stable_X).rank() == len(farm_stable_X.columns)):
    farm_stable_fit = clustered_ols(farm_stable_X, farm_stable['partial_value'])
    display(pd.DataFrame({'all farms': farm_fd[0],
                          'same crop set': farm_stable_fit[0]}).round(3))
    print("Same-crop-set farms:", len(farm_stable))
else:
    print("Too few independent input changes for the same-crop-set fit")

### What changes when we follow farms?



### Read the estimates with the diagnostics



On the checked extracts, the land coefficient is 0.507 in the first
cross-section and 0.886 in the second. Pooling with a wave indicator
gives 0.660; first differences give 0.363. The corresponding pooled and
differenced labor slopes are 0.297 and 0.091, with farm-cluster standard
errors of 0.052 and 0.091. These describe the same 506 farms throughout.

The joint farm-mean test has chi-square 49.335 on two degrees of
freedom (p approximately $1.94\times10^{-11}$), under independent-farm
asymptotics. This rejects the tested pooled restriction; it doesn't
establish the differenced model's exogeneity condition. With only 21
same-crop-set farms, the differenced land slope is 0.631 and the labor
slope is -0.142. That change is another reason to investigate the area
measure, not a reason to select this small sample as the preferred answer.

Time-constant additive effects have been removed. Missing own-consumed
output, heterogeneous prices, changes in management, and mismeasured
inputs have not. The two-period panel gives us a different comparison,
whose assumptions must still be defended.



## Lifecycle consumption



### Why study consumption over a lifetime?



### The life-cycle budget



### An application: inequality as cohorts age



## Panels from Cross-Sections



### Following groups instead of households



### A pseudo-panel



### From cells to a regression



### Identification: age, cohort, and time



### Start with household observations



We need food expenditure, household size, the head's age, and survey
weights. These are the same survey tables used earlier. Ask for food
expenditure in constant-2017 local currency so inflation and the change
in Ghana's currency units aren't mistaken for changes in purchasing.



In [1]:
import warnings
warnings.filterwarnings("ignore", message=r"(?s).*Set LSMS_[A-Z_]+=1 to make this fatal")
warnings.filterwarnings("ignore", message=r"household_characteristics: replaced .*")
warnings.filterwarnings("ignore", message=r"Refusing workspace copy of DVC-tracked file .*")

import lsms_library as ll
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ghana = ll.Country('GhanaLSS')
roster = ghana.household_roster()
chars = ghana.household_characteristics()
sample = ghana.sample()
food = ghana.food_expenditures(numeraire='LCU-real-2017')

First make one observation per household and wave. Sum expenditure over
food items. The characteristics table already provides log household
size, so subtracting it from log expenditure gives log expenditure per
member. This is a per-member comparison, not an adjustment for different
needs of adults and children.

For age, keep households with exactly one recorded head. A missing or
ambiguous head cannot define our cohort. The survey weight belongs to
the household, so the averages below describe *households* classified by
the head's birth year; they aren't person-weighted averages.



In [1]:
heads = roster[roster['Relationship'].str.strip().str.lower().eq('head')]
head_count = heads.groupby(['t', 'i']).size()
age = heads.groupby(['t', 'i'])['Age'].first().rename('age')
age = age[head_count.eq(1)]
expenditure = food['Expenditure'].groupby(['t', 'i']).sum(min_count=1).rename('food')
log_size = chars['log HSize'].groupby(['t', 'i']).first().rename('log_size')
weights = sample['weight'].reorder_levels(['t', 'i']).sort_index().rename('weight')

cohort_data = pd.concat([expenditure, log_size, age, weights], axis=1)
cohort_data = cohort_data.replace([np.inf, -np.inf], np.nan).dropna()
cohort_data = cohort_data[cohort_data['food'].gt(0)
                          & cohort_data['weight'].gt(0)
                          & cohort_data['age'].between(25, 70)].copy()
cohort_data['log_food_pc'] = (np.log(cohort_data['food'].astype(float))
                             - cohort_data['log_size'])
display(cohort_data.groupby('t').agg(households=('age', 'size'),
                                    mean_age=('age', 'mean')).round(1))

We use the final year named in each survey wave as its reference year.
Subtract the head's reported age to approximate the birth year. The
surveys span more than one calendar year and ages can be rounded, so a
head close to a cohort boundary can be misclassified.

Five-year cohorts group estimated birth years 1960–64 together,
1965–69 together, and so on. Dividing by five and taking the floor finds
the group; multiplying by five gives its first birth year.



In [1]:
wave_year = {wave: int(wave[:4]) + 1 for wave in ghana.waves}
cohort_data['year'] = cohort_data.index.get_level_values('t').map(wave_year)
cohort_data['born'] = cohort_data['year'] - cohort_data['age']
cohort_data['cohort'] = (cohort_data['born'] // 5 * 5).astype(int)
display(cohort_data[['age', 'year', 'born', 'cohort']].head())

### Calculate cohort means and dispersion



Let $\ell_i$ be log real food expenditure per member. Within each
cohort-wave cell, calculate
$$
\bar\ell=\frac{\sum_i w_i\ell_i}{\sum_i w_i},\qquad
v=\frac{\sum_i w_i(\ell_i-\bar\ell)^2}{\sum_i w_i}.
$$

The second quantity is a weighted descriptive variance among households.
It isn't the variance of the estimated mean or a regression standard
error. Notice also that we average *logs*, rather than take the log of
average expenditure.

Put these steps in a function so we can change the cohort width without
changing how we calculate the summaries. The final two conditions omit
cells whose birth band is only partly covered by our age range, 25–70.



In [1]:
def cohort_summary(data, width=5):
    d = data.reset_index().copy()
    d['cohort'] = (d['born'] // width * width).astype(int)
    d['weighted_log'] = d['weight'] * d['log_food_pc']
    d['weighted_age'] = d['weight'] * d['age']
    groups = d.groupby(['cohort', 't'])
    total_weight = groups['weight'].sum()
    mean_for_row = (groups['weighted_log'].transform('sum')
                    / groups['weight'].transform('sum'))
    d['weighted_deviation2'] = d['weight'] * (d['log_food_pc'] - mean_for_row)**2
    groups = d.groupby(['cohort', 't'])
    result = pd.DataFrame({
        'mean_log': groups['weighted_log'].sum() / total_weight,
        'var_log': groups['weighted_deviation2'].sum() / total_weight,
        'age': groups['weighted_age'].sum() / total_weight,
        'n': groups.size(),
        'year': groups['year'].first(),
    })
    birth_start = result.index.get_level_values('cohort')
    youngest = result['year'] - (birth_start + width - 1)
    oldest = result['year'] - birth_start
    return result[youngest.ge(25) & oldest.le(70)]

cells = cohort_summary(cohort_data, width=5)
display(cells.head(12).round(3))

### Check the cell sizes



Before drawing a line through cohort summaries, inspect the observations
behind each point. We'll display cells with at least 100 households.
This is a choice for the exercise, not a guarantee of precision: unequal
weights and sampling clusters also affect precision.

A cohort needs at least three displayed waves to give us more than one
change to look at. A blank cell in the count table means that the cohort
isn't fully covered by our age window in that wave.



In [1]:
minimum_cell = 100
display(cells['n'].unstack('t').astype('Int64'))
print("Cells meeting the display cutoff:", int(cells['n'].ge(minimum_cell).sum()),
      "of", len(cells))
big = cells[cells['n'].ge(minimum_cell)].reset_index()
number_of_waves = big.groupby('cohort')['t'].nunique()
shown_cohorts = number_of_waves.index[number_of_waves.ge(3)]
big = big[big['cohort'].isin(shown_cohorts)]

### Follow the same cohorts across surveys



Each line below represents a fixed birth-year band. The left panel shows
average log real food purchases per member; the right shows the variance
of those logs. Read the axes before comparing the shapes: a cohort can
have rising average purchases while differences among its households
also grow.



In [1]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True)
for ax in axes:
    ax.set_prop_cycle(color=plt.cm.tab20.colors)
for cohort, group in big.groupby('cohort'):
    group = group.sort_values('year')
    label = f"{cohort}-{cohort + 4}"
    axes[0].plot(group['age'], group['mean_log'], 'o-', label=label)
    axes[1].plot(group['age'], group['var_log'], 'o-', label=label)
axes[0].set_ylabel('Mean log purchases per member')
axes[1].set_ylabel('Variance of log purchases per member')
for ax in axes:
    ax.set_xlabel('Mean age of household head')
    ax.grid(alpha=.2)
fig.legend(*axes[1].get_legend_handles_labels(), title='Birth cohort',
           loc='lower center', ncol=6, fontsize=8, frameon=False)
fig.tight_layout(rect=[0, .2, 1, 1])
plt.show()

Where a line rises in the right panel, within-cohort dispersion rises
between those surveys. That interval also contains changes in prices,
policy, measurement, and who heads a household. A vertical difference
between lines isn't an identified cohort effect, and overlapping lines
don't establish that cohort effects are absent.

This is a descriptive application inspired by Deaton and Paxson. Food
purchases are narrower than consumption: changing reliance on own-grown
food can change the graph without the same change in living standards.
Dividing by household size also leaves differences in household needs.



### Compare changes and cohort widths



Summarize the change from the first to the last displayed wave for each
cohort. Keep the years and ages beside the change: not every cohort is
observed over the same interval. Dividing by years gives a descriptive
annual change, not an estimated causal age effect.



In [1]:
ordered = big.sort_values(['cohort', 'year'])
first = ordered.groupby('cohort').first()
last = ordered.groupby('cohort').last()
cohort_changes = pd.DataFrame({
    'first_year': first['year'], 'last_year': last['year'],
    'first_age': first['age'], 'last_age': last['age'],
    'variance_change': last['var_log'] - first['var_log'],
})
cohort_changes['change_per_year'] = (cohort_changes['variance_change']
    / (cohort_changes['last_year'] - cohort_changes['first_year']))
display(cohort_changes.round(3))

Here all eleven displayed five-year cohorts have lower dispersion at
their last displayed date than at their first. Several paths rise and
fall in between. These food-purchase profiles don't show a general
increase in dispersion as cohorts age, and they don't by themselves
test a model of total consumption or identify an age effect.

Now widen the birth-year bands to ten years. Larger cells usually reduce
sampling noise, but combine people with more different ages. Repeat the
same calculation and cutoff; don't choose the width because one graph
looks closer to a prediction.



In [1]:
cells10 = cohort_summary(cohort_data, width=10)
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for ax, width, table in zip(axes, [5, 10], [cells, cells10]):
    ax.set_prop_cycle(color=plt.cm.tab20.colors)
    visible = table[table['n'].ge(minimum_cell)].reset_index()
    for cohort, group in visible.groupby('cohort'):
        if group['t'].nunique() >= 3:
            group = group.sort_values('year')
            ax.plot(group['age'], group['var_log'], 'o-',
                    label=f"{cohort}-{cohort + width - 1}")
    ax.set_title(f'{width}-year birth cohorts')
    ax.set_xlabel('Mean age of household head')
    ax.legend(fontsize=8, frameon=False, ncol=3,
              loc='upper center', bbox_to_anchor=(.5, -.2))
    ax.grid(alpha=.2)
axes[0].set_ylabel('Variance of log purchases per member')
fig.tight_layout()
plt.show()
display(pd.DataFrame({
    '5-year': cells['n'].describe(), '10-year': cells10['n'].describe()
}).round(1))

Read the two graphs together. Do they support a general rise in
within-cohort food-purchase dispersion, or do some paths fall? Which
changes coincide across cohorts at a survey date? The exercise provides
these comparisons; separating a life-cycle mechanism from period changes
would require additional assumptions and a broader consumption measure.



## Household panels and welfare



### The Uganda panel



In [1]:
uga = ll.Country('Uganda')
print(uga.waves)
print(uga.data_scheme)

In [1]:
ufood = uga.food_expenditures()
uc = ufood.groupby(['t', 'i']).sum().squeeze()
uc = np.log(uc.where(uc > 0)).rename('logc')

un = uga.household_characteristics().sum(axis=1).groupby(['t', 'i']).first()
un = np.log(un.astype(float).where(un > 0)).rename('logn')

p = pd.concat([uc, un], axis=1).dropna()
p = p[np.isfinite(p).all(axis=1)]

# How many households are actually observed more than once?
counts = p.groupby('i').size()
print(counts.value_counts().sort_index())

In [1]:
# Keep only households seen more than once -- singletons contribute nothing
# to a within estimator, and quietly inflate the apparent sample size.
p = p[counts.reindex(p.index.get_level_values('i')).to_numpy() > 1]

# Time dummies, then demean everything within household.  After demeaning
# there is no constant to include.
T = pd.get_dummies(p.index.get_level_values('t'), prefix='t',
                   drop_first=True, dtype=float)
T.index = p.index
q = pd.concat([p, T], axis=1)
qd = q - q.groupby(level='i').transform('mean')

# The farm application's estimator, reused: least squares with a CR1
# covariance clustered on the index level `i' -- here the household.
b, V = clustered_ols(qd.drop(columns='logc'), qd.logc)
print(coef_table(b, V).to_string())

Compare the coefficient on `logn` here with the cross-sectional one from
session 4.  If they differ, the cross-sectional estimate was contaminated by
whatever it is about large households that also predicts consumption.



### The same panel, in welfare units



Everything above treats $\log c$ as the outcome.  Session 4 argued that
$w = -\log\lambda$ is the better welfare measure, because prices and
household composition are swept into the good-time effects and the Barten
scales rather than left in the number.  Here is the same regression on
$w$, so you can see how much it matters.



In [1]:
from pathlib import Path
import lsms_library as ll
import cfe
from cfe import Regression
import numpy as np, pandas as pd

def uganda_cfe():
    # The estimated CFE system for Uganda, as a cfe.Regression.  The object
    # carries beta, gamma, w = -log lambda, and predicted expenditures, so
    # nothing downstream has to refit.  Uses a copy staged on the hub if
    # there is one, else a copy you estimated earlier, else estimates it
    # from scratch (about forty seconds) and caches the result.  So this
    # cell is self-contained: no other notebook need have been run first.
    staged = Path('/srv/data/hhsurveys/uganda.rgsn')
    mine   = Path.home() / '.cache' / 'hhsurveys' / 'uganda.rgsn'
    for c in (staged, mine):
        if c.exists():
            return cfe.read_pickle(str(c))

    uga = ll.Country('Uganda')
    x = uga.food_expenditures().squeeze()
    agg = (uga.categorical_mapping['harmonize_food']
              .set_index('Preferred Label')['Aggregate Label'].to_dict())
    x = x.rename(index=agg, level='j')
    x = x.groupby(x.index.names).sum()

    y = np.log(x.replace(0, np.nan).dropna()).groupby(['i', 't', 'j']).sum()
    y = pd.concat({1: y}, names=['m']).reorder_levels(['i', 't', 'm', 'j']).sort_index()

    d0 = uga.household_characteristics()
    d = d0.assign(
        Girls=d0[[f'F {a}' for a in ['00-03', '04-08', '09-13', '14-18']]].sum(axis=1),
        Boys =d0[[f'M {a}' for a in ['00-03', '04-08', '09-13', '14-18']]].sum(axis=1),
        Women=d0[[f'F {a}' for a in ['19-30', '31-50', '51+']]].sum(axis=1),
        Men  =d0[[f'M {a}' for a in ['19-30', '31-50', '51+']]].sum(axis=1),
    )[['Girls', 'Boys', 'Women', 'Men', 'log HSize']].dropna(how='any')
    d = d.groupby(['i', 't']).first()
    d = pd.concat({1: d}, names=['m']).reorder_levels(['i', 't', 'm']).sort_index()

    r = Regression(y=y, d=d)
    r.get_beta(); r.get_w(); r.predicted_expenditures()
    mine.parent.mkdir(parents=True, exist_ok=True)
    r.to_pickle(str(mine))
    return r

r = uganda_cfe()
w = r.get_w()
w.groupby('t').agg(['size', 'mean', 'std']).round(3)

In [1]:
# The same within estimator, in welfare units.
# w arrives indexed (i, t, m) while p is (t, i).  Line them up explicitly:
# concat on a differently-ordered MultiIndex does not raise, it just fails
# to align -- the same trap that emptied the pseudo-panel above.
wi = w.droplevel('m') if 'm' in (w.index.names or []) else w
wi = wi.rename('w').reorder_levels(['t', 'i']).sort_index()

q2 = pd.concat([wi, p.logn], axis=1).dropna()
assert len(q2) > 0.5 * min(len(wi), len(p)), "w and p failed to align"

c2 = q2.groupby('i').size()
q2 = q2[c2.reindex(q2.index.get_level_values('i')).to_numpy() > 1]

T2 = pd.get_dummies(q2.index.get_level_values('t'), prefix='t',
                    drop_first=True, dtype=float)
T2.index = q2.index
Q2 = pd.concat([q2, T2], axis=1)
Q2d = Q2 - Q2.groupby(level='i').transform('mean')

b_w, V_w = clustered_ols(Q2d.drop(columns='w'), Q2d.w)
print(coef_table(b_w, V_w).to_string())

Three comparisons to make, and the third is the one to remember.

The coefficient on `logn`.  In $\log c$ it is about $+0.34$: a bigger
household simply spends more.  In $w$ it is about $-0.03$ — small and
*negative*.  Household size has already been absorbed into the Barten
scales, so what is left is what size does to welfare net of need, and that
is slightly adverse.  The two numbers are not rival estimates of one
parameter; they answer different questions.

The time effects.  In $\log c$ they climb steeply and monotonically, from
0.40 to 1.20 across the panel.  That is mostly inflation, since nothing
deflated those expenditures.  In $w$ they sit within $\pm 0.15$ and do
not trend, because prices live in $a^j_t$ and never entered $w$ at
all.  No CPI was used to achieve that, and none was available.

Now look at 2009–10 in the $w$ column.  It is the only negative time
effect in the panel.  That is the 2008 food price crisis, and it is
invisible in $\log c$, where 2009–10 is simply another step up the
inflation ladder.  Session 6 takes that observation and asks what it does to
the poverty rate.



### Exercise 1: ten-year cohorts



Rebuild the Ghana pseudo-panel with ten-year cohorts. How much do the
lifecycle profiles change? Which conclusion is robust?



### Exercise 2: means of logs, logs of means



Compare the weighted mean of log purchases with the log of weighted mean
purchases in each cohort. Why are they different?



### Exercise 3: attrition in Uganda



What fraction of the 2009–10 households appear in 2011–12? Are the
leavers different in 2009–10 consumption from the stayers? What does
that do to your fixed-effects estimate?



### Exercise 4: Mundlak (1978) on the farms



Mundlak's later formulation augments the *levels* regression with farm
means of the regressors. Add them, keep the wave indicator, and show the
slopes reproduce the first differences — an algebra check on the
implementation, good to machine precision.

Then test the mean terms jointly with the farm-cluster covariance. What
does rejection establish, and what does it not?



### Exercise 5: FWL on an unbalanced panel



Redo the within estimator by Frisch–Waugh–Lovell on an unbalanced panel
of your own making: eight farms, four dates, a few rows deleted. Use
`DataMat.resid` to partial the farm and wave indicators out of the outcome
and the regressors *jointly*, then regress the residuals.

Check it against full indicator OLS, and check that one pass of
subtracting farm means and wave means does *not* agree. Why does balance
matter here when it didn't in the balanced case?
(`metrics_miscellany.estimators.fwl_regression` partials recursively.)



### Exercise 6: put the purchased inputs back



Purchased inputs sit in $u$. Add $\log$ production-input spend as a
third regressor. Only 256 of the 506 farms report positive spend in both
waves, so re-estimate the two-input model on those same 256 as well —
three sets of slopes, not two.

How much of the movement in $\beta_H$ and $\beta_L$ is the omitted
input, and how much is the change of sample? And what must be true of
input prices for the new coefficient to be an elasticity rather than a
description?

